> **Disclaimer:** This notebook is for *educational and research training purposes only*. It does not constitute medical, clinical, or diagnostic advice. All computational predictions (pLDDT, RMSD, affinity scores) are approximations from machine learning models and must not be treated as experimental results. See the full [DISCLAIMER](https://github.com/JobAiReady/lagos-bio-design/blob/main/DISCLAIMER.md) and [PRIVACY POLICY](https://github.com/JobAiReady/lagos-bio-design/blob/main/PRIVACY_POLICY.md).

# Module 1: The New Paradigm — From Yaba to Stockholm
## Lab: Environment Setup & First Protein Visualization

**Lagos Bio-Design Bootcamp** | [JobAiReady](https://github.com/JobAiReady/lagos-bio-design)

---

### Objective
Initialize your cloud-based bio-design environment and visualize your first protein structure.

### Prerequisites
- Google Account (for Colab)
- Basic Python knowledge

### Deliverable
A screenshot of your interactive 3D protein visualization colored by confidence score.

---
## Background & Key Concepts

In October 2024, the Nobel Prize in Chemistry was awarded to **David Baker** (University of Washington) for computational protein design, and **Demis Hassabis & John Jumper** (Google DeepMind) for **AlphaFold** — an AI that predicts the 3D structure of virtually every known protein. This marked the shift from *discovering* proteins in nature to *creating* them from scratch.

**Proteins** are molecular machines made of chains of amino acids. The sequence of amino acids determines how the chain folds into a specific 3D shape, and the shape determines function. For decades, scientists relied on **Directed Evolution** — randomly mutating natural proteins and selecting the best variants. The new paradigm, **De Novo Design**, uses AI to generate proteins that have never existed in nature.

### Why This Matters for Lagos

Africa's disease burden (Lassa fever, Malaria, Tuberculosis) demands novel protein-based diagnostics and therapeutics. The tools to design these — AlphaFold, RFDiffusion, ProteinMPNN — are **open-source** and run on cloud GPUs. A laptop in Yaba now has the same design capability as a lab at MIT.

### Key Terms

| Term | Definition |
|------|-----------|
| **Protein** | A molecular machine made of amino acids that folds into a specific 3D shape to perform a biological function |
| **AlphaFold** | Google DeepMind's AI that predicts protein 3D structure from amino acid sequence |
| **pLDDT** | Predicted Local Distance Difference Test — AlphaFold's confidence score (0–100). >90 = high confidence, <50 = disordered |
| **De Novo Design** | Creating entirely new proteins from scratch using AI, rather than modifying natural ones |
| **PDB** | Protein Data Bank — the global repository of experimentally determined 3D protein structures |

---
## Step 1: Environment Setup
Clone the repository and install dependencies in this Colab environment.

In [ ]:
# Clone the Lagos Bio-Design Bootcamp repository
!git clone https://github.com/JobAiReady/lagos-bio-design.git
%cd lagos-bio-design

In [ ]:
# Install core dependencies for protein analysis
!pip install -q biopython matplotlib numpy pandas py3Dmol
!pip install -q colabfold[alphafold2] --no-deps 2>/dev/null || echo 'ColabFold optional — see instructions below'

In [ ]:
# Verify installation
import numpy as np
import matplotlib.pyplot as plt
from Bio import SeqIO

print('Environment ready!')
print(f'NumPy: {np.__version__}')

---
## Step 2: AlphaFold Inference — Predict Insulin Structure
We'll use ColabFold to predict the 3D structure of human insulin from its amino acid sequence.

> **Note:** If ColabFold is not available, you can download a pre-computed insulin structure from the [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/entry/P01308).

In [ ]:
# Human Insulin sequence (Chain A + Chain B)
insulin_sequence = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"

# Write to FASTA file
with open('insulin.fasta', 'w') as f:
    f.write(f'>human_insulin\n{insulin_sequence}\n')

print(f'Sequence length: {len(insulin_sequence)} residues')
print(f'Sequence written to insulin.fasta')

In [ ]:
# Option A: Run ColabFold prediction (requires GPU runtime)
# Uncomment below if ColabFold installed successfully:

# from colabfold.batch import run_batch
# run_batch(input_sequences='insulin.fasta', output_dir='./results')

# Option B: Download pre-computed structure from AlphaFold DB
import urllib.request
import os

os.makedirs('results', exist_ok=True)
url = 'https://alphafold.ebi.ac.uk/files/AF-P01308-F1-model_v4.pdb'
urllib.request.urlretrieve(url, 'results/insulin_predicted.pdb')
print('Downloaded insulin structure from AlphaFold DB')

---
## Step 3: Analyze pLDDT Confidence Scores
Let's analyze the confidence scores programmatically before visualizing the structure in 3D.

In [ ]:
# Parse the PDB and extract pLDDT scores (stored in B-factor column)
from Bio.PDB import PDBParser
import numpy as np
import matplotlib.pyplot as plt

parser = PDBParser(QUIET=True)
structure = parser.get_structure('insulin', 'results/insulin_predicted.pdb')

# Extract pLDDT (B-factor) per residue
plddt_scores = []
for model in structure:
    for chain in model:
        for residue in chain:
            if residue.get_id()[0] == ' ':  # Skip hetero atoms
                ca = residue['CA'] if 'CA' in residue else list(residue.get_atoms())[0]
                plddt_scores.append(ca.get_bfactor())

plddt = np.array(plddt_scores)
print(f'Residues: {len(plddt)}')
print(f'Mean pLDDT: {plddt.mean():.1f}')
print(f'Min pLDDT:  {plddt.min():.1f}')
print(f'Max pLDDT:  {plddt.max():.1f}')

In [ ]:
# Plot pLDDT per residue
plt.figure(figsize=(12, 4))
plt.bar(range(len(plddt)), plddt, color=[
    '#0053D6' if s > 90 else '#65CBF3' if s > 70 else '#FFDB13' if s > 50 else '#FF7D45'
    for s in plddt
])
plt.axhline(y=90, color='blue', linestyle='--', alpha=0.3, label='Very high (>90)')
plt.axhline(y=70, color='cyan', linestyle='--', alpha=0.3, label='Confident (>70)')
plt.axhline(y=50, color='yellow', linestyle='--', alpha=0.3, label='Low (>50)')
plt.xlabel('Residue Index')
plt.ylabel('pLDDT Score')
plt.title('AlphaFold Confidence (pLDDT) — Human Insulin')
plt.legend()
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

---
## Step 4: Interactive 3D Visualization (In-Browser)

We use **py3Dmol** to render the protein structure directly in Colab — no local software needed. The structure is colored by pLDDT confidence using the same AlphaFold color scheme.

In [ ]:
import py3Dmol

# Read the PDB file
with open('results/insulin_predicted.pdb', 'r') as f:
    pdb_data = f.read()

# Create interactive 3D viewer
view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, 'pdb')

# Color by pLDDT (B-factor) using AlphaFold color scheme
view.setStyle({'cartoon': {'colorscheme': {
    'prop': 'b',
    'gradient': 'rwb',
    'min': 50,
    'max': 90
}}})

view.zoomTo()
view.setBackgroundColor('white')
view.show()

print('🔵 Blue = High confidence (pLDDT > 90)')
print('⚪ White = Medium confidence')
print('🔴 Red = Low confidence (pLDDT < 50)')
print('\nRotate: click + drag | Zoom: scroll | Pan: right-click + drag')

---
## Summary

In this lab you:
1. Set up a cloud-based Python environment for bio-design
2. Retrieved/predicted a protein structure using AlphaFold
3. Analyzed pLDDT confidence scores programmatically
4. Visualized the structure in interactive 3D, colored by confidence

**Next:** Module 2 — Building the RFDiffusion → ProteinMPNN pipeline

---

> **Optional — For Students Who Want More:**
> [PyMOL](https://pymol.org/) is the industry-standard molecular visualization tool used in pharma and biotech labs. If you'd like to explore publication-quality rendering, ray tracing, and advanced structural analysis, install PyMOL locally and try these commands:
> ```
> load insulin_predicted.pdb
> spectrum b, rainbow_rev, minimum=50, maximum=90
> set cartoon_fancy_helices, 1
> bg_color white
> ray 1200, 900
> png insulin_plddt.png
> ```
> Download the PDB file below to use with PyMOL or any other local viewer.